# 👥 Módulo 16 — Múltiples Agentes (Coordinación Manual)

> **Objetivo:** Coordinar varios agentes especializados sin usar todavía el framework de workflows.

## ¿Por qué múltiples agentes?

Un solo agente con instrucciones muy largas es difícil de mantener y suele dar peores resultados.
Dividir tareas entre **agentes especializados** mejora calidad, modularidad y observabilidad.

## Patrones que veremos

| Patrón | Diagrama |
|--------|----------|
| **Secuencial** | `Escritor → Crítico` |
| **Pipeline** | `Redactor → Traductor → Resumidor` |
| **Fan-out** | `mismo input → [Optimista, Pesimista, Neutral]` (en paralelo con `asyncio.gather`) |

En este módulo orquestamos **manualmente** con `agent.run()`.
En el **Módulo 17** veremos cómo el framework lo hace por ti con `SequentialBuilder` / `ConcurrentBuilder`.


## ⚙️ Setup inicial

Esta celda carga la configuración de Azure OpenAI desde `appsettings.Development.json`
(o variables de entorno) y crea el cliente que reutilizaremos durante todo el notebook.

> 💡 Solo necesitas ejecutar esta celda **una vez** al abrir el notebook.


In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
# Si abriste el notebook desde la carpeta notebooks/, sube un nivel
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados. Endpoint:", create_chat_client().__class__.__name__)


## 1️⃣ Secuencial — Escritor → Crítico

El escritor genera contenido. La salida se envía como input al crítico, que la evalúa.


In [ ]:
writer = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un escritor creativo. Escribe un párrafo corto (máximo 3 oraciones) "
        "sobre el tema proporcionado. Sé conciso y creativo."
    ),
    name="Escritor",
)

critic = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un crítico literario estricto. Evalúa el texto proporcionado en una "
        "oración breve indicando un aspecto positivo y uno a mejorar. Sé directo."
    ),
    name="Critico",
)

writer_response = await writer.run(
    "Escribe sobre la inteligencia artificial en la vida cotidiana"
)
print("📝 Escritor:")
print(f"   {writer_response.text}\n")

critic_response = await critic.run(f'Evalúa este texto: "{writer_response.text}"')
print("🔍 Crítico:")
print(f"   {critic_response.text}")


## 2️⃣ Pipeline — Redactor → Traductor → Resumidor

Tres agentes encadenados. Cada uno tiene un rol muy específico.


In [ ]:
redactor = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un redactor técnico. Escribe una explicación breve (2-3 oraciones) "
        "sobre el concepto proporcionado. Usa lenguaje claro y preciso. Responde en español."
    ),
    name="Redactor",
)

translator = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un traductor profesional. Traduce el texto proporcionado al inglés. "
        "Solo responde con la traducción, sin agregar notas ni explicaciones."
    ),
    name="Traductor",
)

summarizer = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un resumidor. Crea un resumen de una sola oración del texto proporcionado. "
        "Sé extremadamente conciso. Responde en español."
    ),
    name="Resumidor",
)

topic = "¿Qué es un contenedor Docker?"

r1 = await redactor.run(topic)
print(f"📝 Redactor (ES):\n   {r1.text}\n")

r2 = await translator.run(r1.text)
print(f"🌐 Traductor (EN):\n   {r2.text}\n")

r3 = await summarizer.run(r2.text)
print(f"📋 Resumidor:\n   {r3.text}")


## 3️⃣ Fan-out — múltiples perspectivas en paralelo

Tres agentes analizan el MISMO input con perspectivas distintas. `asyncio.gather` los corre en paralelo.


In [ ]:
import asyncio

perspectives = [
    ("Optimista", "Eres un analista extremadamente optimista. Analiza el mensaje destacando solo lo positivo en una oración."),
    ("Pesimista", "Eres un analista muy pesimista. Analiza el mensaje señalando solo riesgos o problemas en una oración."),
    ("Neutral",   "Eres un analista neutral y equilibrado. Analiza el mensaje de forma objetiva en una oración."),
]

input_message = (
    "Una empresa de tecnología planea reemplazar el 50% de sus procesos manuales con "
    "inteligencia artificial"
)

agents = [Agent(client=create_chat_client(), instructions=instr, name=name) for name, instr in perspectives]

# 🚀 Ejecución en paralelo
responses = await asyncio.gather(*(a.run(input_message) for a in agents))

print(f'📨 Mensaje: "{input_message}"\n')
for (name, _), resp in zip(perspectives, responses):
    print(f"🔹 {name}: {resp.text}")


## 🎯 Resumen

| Patrón | Ventaja | Coste |
|--------|---------|-------|
| Secuencial | Simple, fácil de razonar | Latencia = suma de cada paso |
| Pipeline | Roles muy especializados | Igual que secuencial |
| Fan-out con `asyncio.gather` | Paralelismo real | Latencia = el más lento |

> 💡 En el **Módulo 17** veremos el equivalente "framework" (`SequentialBuilder`, `ConcurrentBuilder`)
> que ofrece eventos, observabilidad y agregación automática.

**Siguiente módulo →** [Módulo 17 — Agentes en Workflows](./17_agents_in_workflows.ipynb)
